In [4]:
import pandas as pd
import numpy as np
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

In [8]:
n_items_list = [50, 100, 100, 100, 500, 500, 1000, 1000, 1000, 1500, 1500]
m_users_list = [50, 100, 300, 500, 100, 500, 100, 200, 300, 1000, 2000]

for n_items, m_users in zip(n_items_list, m_users_list):
    # Your code using n_items and m_users
    # For example:
    print(f"n_items: {n_items}, m_users: {m_users}")
    # Run your analysis or processing using n_items and m_users here


n_items: 50, m_users: 50
n_items: 100, m_users: 100
n_items: 100, m_users: 300
n_items: 100, m_users: 500
n_items: 500, m_users: 100
n_items: 500, m_users: 500
n_items: 1000, m_users: 100
n_items: 1000, m_users: 200
n_items: 1000, m_users: 300
n_items: 1500, m_users: 1000
n_items: 1500, m_users: 2000


In [19]:
#main loop 

k = 10 # k-fold validation 
n_items_list = [100, 100,500,500,500,1000,1000,1000]
m_users_list = [50, 700, 50, 300,700,50,  500, 700]
random_seed = 20230808

#prepare the result summary table 
result_summary_df = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','std_acc','avg_rmse','std_rmse','avg_mad','std_mad','avg_time','std_time'])

# 6개의 열을 가진 빈 DataFrame 생성
result_detail_df = pd.DataFrame(columns=['n_items','m_users','acc', 'rmse', 'mad', 'time'])
#main 

for n, m in zip(n_items_list, m_users_list):
    n_items = n 
    n_users = m 

    ten = [str(j).zfill(2) for j in list(range(1,k+1))]
    
    labs = ['test'+i for i in ten]
    labs.insert(0,'orig')
    
    url_dict = {}
    url_dict["orig"] = './data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
    
    for i in ten:
        url_dict["test{0}".format(i)] = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'
    print(labs)
    
    for lab in labs:
        print(url_dict[lab])    
    
    df_dict = {}
    for lab in labs:
        #print(lab)
        df = pd.read_csv(url_dict[lab])
        #print(lab)
        df_dict[lab] = df.set_index('item_id')

    #compute the sparsity 
    n_row = df_dict["orig"].shape[0]
    n_col = df_dict["orig"].shape[1]
    Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
    num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
    sparsity = 1 - num_Obs / (n_row * n_col)
    print("sparsity: "+str(sparsity))

    #Run algorithm 
    acc_list = []
    rmse_list = []
    mad_list = []
    time_list = []

    df_orig = df_dict["orig"]
    df_orig.columns = range(df_orig.shape[1])

    mm = df_orig.shape[0]
    nn = df_orig.shape[1]

    labs_test = labs[1:]

    col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
    count = 0 

    for lab in labs_test:
        count += 1
        #print(lab)
        
        df = df_dict[lab]
        
        masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
        #df, maxs = normal1(masked_df)
        #df, maxs = normal2(masked_df)
        #df, maxs, probs = normal3(masked_df)

        # from missingpy import MissForest
        # imputer = MissForest()

        from fancyimpute import SoftImpute
        imputer = SoftImpute(verbose=False)

        start = time_lib.time()
        imputed = pd.DataFrame(imputer.fit_transform(df))

        # Rounding for when no normalization           
        for i in range(mm):
            for j in range(nn):
                x = imputed.iloc[i,j]
                if x < 1:
                    imputed.iloc[i,j] = 1
                elif x > col_max[j]:
                    imputed.iloc[i,j] = col_max[j]
                else:
                    imputed.iloc[i,j] = round(x,0)

        #imputed = unnormal1(imputed, maxs)
        #imputed = unnormal2(imputed, maxs)
        #imputed = unnormal3(imputed, maxs, probs)

        end = time_lib.time()
        time = end - start

        df_orig.index = range(mm)
        imputed.index = range(mm)

        df_orig.columns = range(nn)
        imputed.columns = range(nn)

        #save the result data
        url = './result/'

        if count<10:
            filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_soft_imputed.csv'
        else:
            filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_soft_imputed.csv'
        
        imputed.to_csv(filename)
        print("saved the result file as "+str(filename))
        diff = df_orig - imputed
        sq_diff = diff ** 2
        abs_diff = abs(diff)

        n_match = 0
        for i in range(mm):
            for j in range(nn):
                if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
                    n_match += 1
        acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
        rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
        mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
        
        acc_list.append(acc)
        rmse_list.append(rmse)
        mad_list.append(mad)
        time_list.append(time)

    df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
    display("n: "+str(n_items)+" ,m: "+ str(n_users))
    display(df.T)
    df_temp = df.T
    df_temp_2 = df_temp.copy()
    a = [n] * k 
    b = [m] * k
    df_temp_2.rename(columns={df_temp_2.columns[0]: 'acc', df_temp_2.columns[1]: 'rmse', df_temp_2.columns[2]: 'mad',df_temp_2.columns[3]: 'time'}, inplace=True)
    df_temp_2.insert(0,'m_users',b)
    df_temp_2.insert(0,'n_items',a)
    

    # result_detail_df와 df_temp을 행 방향으로 이어 붙임
    result_detail_df = pd.concat([result_detail_df, df_temp_2], axis=0)
    
    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data_all_soft_imputed.csv'
    df_temp.to_csv(filename)
    print('saved '+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+ 'the summary result file as' +str(filename))
    # 열별 평균값 계산
    mean_values = df_temp.mean()
    # 열별 표준편차 계산
    std_values = df_temp.std()
    #new_row = [n,m,sparsity]+mean_values.tolist()
    new_row = [n,m,sparsity,mean_values[0],std_values[0],mean_values[1],std_values[1],mean_values[2],std_values[2],mean_values[3],std_values[3]]
    #display(new_row)
    result_summary_df = result_summary_df.append(pd.Series(new_row, index=result_summary_df.columns), ignore_index=True)
    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'soft_imputed.csv'
    df_temp.to_csv(filename)
    print("saved the summary result file as "+str(filename))
    display(result_summary_df)


display(result_summary_df)
display(result_detail_df)

result_summary_df.to_csv('result_summary_df_soft.csv')
result_detail_df.to_csv('result_detail_df_soft.csv')
        


        

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/100-by-50_original_matrix.csv
./data/20230808_100-by-50_10_fold_test_data01.csv
./data/20230808_100-by-50_10_fold_test_data02.csv
./data/20230808_100-by-50_10_fold_test_data03.csv
./data/20230808_100-by-50_10_fold_test_data04.csv
./data/20230808_100-by-50_10_fold_test_data05.csv
./data/20230808_100-by-50_10_fold_test_data06.csv
./data/20230808_100-by-50_10_fold_test_data07.csv
./data/20230808_100-by-50_10_fold_test_data08.csv
./data/20230808_100-by-50_10_fold_test_data09.csv
./data/20230808_100-by-50_10_fold_test_data10.csv
sparsity: 0.23939999999999995
saved the result file as ./result/20230808_100-by-50_10_fold_test_data01_soft_imputed.csv
saved the result file as ./result/20230808_100-by-50_10_fold_test_data02_soft_imputed.csv
saved the result file as ./result/20230808_100-by-50_10_fold_test_data03_soft_imputed.csv
saved the result file as ./result/20230808_100-by-50_1

'n: 100 ,m: 50'

,0,1,2,3
0,0.426316,0.963819,0.681579,0.323188
1,0.465789,0.911910,0.631579,0.321631
2,0.455263,0.954215,0.657895,0.316415
3,0.407895,0.914791,0.673684,0.306331
4,0.402632,0.977376,0.713158,0.319976
5,0.444737,0.933302,0.655263,0.310827
6,0.450000,0.976028,0.673684,0.309542
7,0.425197,0.925010,0.666667,0.312843
8,0.393701,0.981455,0.716535,0.305843
9,0.451444,0.864888,0.611549,0.313358


saved 20230808_100-by-50the summary result file as./result/20230808_100-by-50_10_fold_test_data_all_soft_imputed.csv
saved the summary result file as ./result/20230808_100-by-50_soft_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,100.0,50.0,0.2394,0.432297,0.0248,0.940279,0.037295,0.668159,0.032357,0.313995,0.006157


...

saved 20230808_1000-by-700the summary result file as./result/20230808_1000-by-700_10_fold_test_data_all_soft_imputed.csv
saved the summary result file as ./result/20230808_1000-by-700_soft_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,100.0,50.0,0.239400,0.432297,0.024800,0.940279,0.037295,0.668159,0.032357,0.313995,0.006157
1,100.0,700.0,0.609329,0.425056,0.007391,0.950217,0.009034,0.677807,0.008760,8.534618,0.195432
2,500.0,50.0,0.466640,0.410381,0.012085,0.964138,0.018580,0.698137,0.018210,1.842386,0.028448
3,500.0,300.0,0.661573,0.439780,0.006749,0.920488,0.009117,0.651525,0.009179,18.280524,0.461281
4,500.0,700.0,0.792409,0.424350,0.005266,0.946208,0.009187,0.677347,0.007820,53.570397,0.676427
5,1000.0,50.0,0.633380,0.406252,0.013785,1.000469,0.017040,0.720582,0.018956,3.586962,0.075132
6,1000.0,500.0,0.837624,0.421380,0.004641,0.967041,0.007045,0.691260,0.006511,65.505906,1.145332
7,1000.0,700.0,0.871296,0.412163,0.004893,0.993336,0.005780,0.712919,0.005009,104.574395,0.949224


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,100.0,50.0,0.239400,0.432297,0.024800,0.940279,0.037295,0.668159,0.032357,0.313995,0.006157
1,100.0,700.0,0.609329,0.425056,0.007391,0.950217,0.009034,0.677807,0.008760,8.534618,0.195432
2,500.0,50.0,0.466640,0.410381,0.012085,0.964138,0.018580,0.698137,0.018210,1.842386,0.028448
3,500.0,300.0,0.661573,0.439780,0.006749,0.920488,0.009117,0.651525,0.009179,18.280524,0.461281
4,500.0,700.0,0.792409,0.424350,0.005266,0.946208,0.009187,0.677347,0.007820,53.570397,0.676427
5,1000.0,50.0,0.633380,0.406252,0.013785,1.000469,0.017040,0.720582,0.018956,3.586962,0.075132
6,1000.0,500.0,0.837624,0.421380,0.004641,0.967041,0.007045,0.691260,0.006511,65.505906,1.145332
7,1000.0,700.0,0.871296,0.412163,0.004893,0.993336,0.005780,0.712919,0.005009,104.574395,0.949224


,n_items,m_users,acc,rmse,mad,time
0,100,50,0.426316,0.963819,0.681579,0.323188
1,100,50,0.465789,0.911910,0.631579,0.321631
2,100,50,0.455263,0.954215,0.657895,0.316415
3,100,50,0.407895,0.914791,0.673684,0.306331
4,100,50,0.402632,0.977376,0.713158,0.319976
...,...,...,...,...,...,...
5,1000,700,0.419136,0.999112,0.710956,104.956039
6,1000,700,0.407925,0.993597,0.716395,105.404133
7,1000,700,0.414983,1.001940,0.715094,103.618505
8,1000,700,0.412209,0.991025,0.711987,105.045271


: 